In [1]:
#  This code is used just to create the skew-T plot of global, annual mean air temperature
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from metpy.plots import SkewT
ncep_url = "http://www.esrl.noaa.gov/psd/thredds/dodsC/Datasets/ncep.reanalysis.derived/"
ncep_air = xr.open_dataset( ncep_url + "pressure/air.mon.1981-2010.ltm.nc", use_cftime=True)
#  Take global, annual average 
coslat = np.cos(np.deg2rad(ncep_air.lat))
weight = coslat / coslat.mean(dim='lat')
Tglobal = (ncep_air.air * weight).mean(dim=('lat','lon','time'))

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [2]:
def make_skewT():
    fig = plt.figure(figsize=(9, 9))
    skew = SkewT(fig, rotation=30)
    skew.plot(Tglobal.level, Tglobal, color='black', linestyle='-', linewidth=2, label='Observations')
    skew.ax.set_ylim(1050, 10)
    skew.ax.set_xlim(-90, 45)
    # Add the relevant special lines
    skew.plot_dry_adiabats(linewidth=0.5)
    skew.plot_moist_adiabats(linewidth=0.5)
    #skew.plot_mixing_lines()
    skew.ax.legend()
    skew.ax.set_xlabel('Temperature (degC)', fontsize=14)
    skew.ax.set_ylabel('Pressure (hPa)', fontsize=14)
    return skew

In [3]:
skew = make_skewT()

NameError: name 'plt' is not defined

In [4]:
#  Load the model output as we have done before
cesm_data_path = "http://thredds.atmos.albany.edu:8080/thredds/dodsC/CESMA/"
atm_control = xr.open_dataset(cesm_data_path + "cpl_1850_f19/concatenated/cpl_1850_f19.cam.h0.nc")
#  The specific humidity is stored in the variable called Q in this dataset:
print(atm_control.Q)

NameError: name 'xr' is not defined

In [5]:
# Take global, annual average of the specific humidity
weight_factor = atm_control.gw / atm_control.gw.mean(dim='lat')
Qglobal = (atm_control.Q * weight_factor).mean(dim=('lat','lon','time'))
# Take a look at what we just calculated ... it should be one-dimensional (vertical levels)
print(Qglobal)

NameError: name 'atm_control' is not defined

In [6]:
fig, ax = plt.subplots()
#  Multiply Qglobal by 1000 to put in units of grams water vapor per kg of air
ax.plot(Qglobal*1000., Qglobal.lev)
ax.invert_yaxis()
ax.set_ylabel('Pressure (hPa)')
ax.set_xlabel('Specific humidity (g/kg)')
ax.grid()

NameError: name 'plt' is not defined

In [7]:
import climlab
#  Make a model on same vertical domain as the GCM
mystate = climlab.column_state(lev=Qglobal.lev, water_depth=2.5)
print(mystate)

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [8]:
radmodel = climlab.radiation.RRTMG(name='Radiation (all gases)',  # give our model a name!
                              state=mystate,   # give our model an initial condition!
                              specific_humidity=Qglobal.values,  # tell the model how much water vapor there is
                              albedo = 0.25,  # this the SURFACE shortwave albedo
                              timestep = climlab.constants.seconds_per_day,  # set the timestep to one day (measured in seconds)
                             )
print(radmodel)

NameError: name 'climlab' is not defined

In [9]:
#  Here's the state dictionary we already created:
radmodel.state

NameError: name 'radmodel' is not defined

In [10]:
#  Here are the pressure levels in hPa
radmodel.lev

NameError: name 'radmodel' is not defined

In [11]:
radmodel.absorber_vmr

NameError: name 'radmodel' is not defined

In [12]:
#  E.g. the CO2 content (a well-mixed gas) in parts per million
radmodel.absorber_vmr['CO2'] * 1E6

NameError: name 'radmodel' is not defined

In [13]:
# here is the data you need for the plot, as a plain numpy arrays:
print(radmodel.lev)
print(radmodel.absorber_vmr['O3'])

NameError: name 'radmodel' is not defined

In [14]:
#  specific humidity in kg/kg, on the same pressure axis
print(radmodel.specific_humidity)

NameError: name 'radmodel' is not defined

In [15]:
for item in radmodel.input:
    print(item)

NameError: name 'radmodel' is not defined

In [16]:
#  This is the fractional area covered by clouds in our column:
radmodel.cldfrac

NameError: name 'radmodel' is not defined

In [17]:
radmodel.Ts

NameError: name 'radmodel' is not defined

In [18]:
radmodel.Tatm

NameError: name 'radmodel' is not defined

In [19]:
radmodel.step_forward()

NameError: name 'radmodel' is not defined

In [20]:
radmodel.Ts

NameError: name 'radmodel' is not defined

In [21]:
climlab.to_xarray(radmodel.diagnostics)

NameError: name 'climlab' is not defined

In [22]:
climlab.to_xarray(radmodel.LW_flux_up)

NameError: name 'climlab' is not defined

In [23]:
radmodel.lev

NameError: name 'radmodel' is not defined

In [24]:
radmodel.lev_bounds

NameError: name 'radmodel' is not defined

In [25]:
radmodel.LW_flux_up[-1]

NameError: name 'radmodel' is not defined

In [26]:
sigma = 5.67E-8
sigma * 288**4

390.0793946112

In [27]:
radmodel.LW_flux_up[0]

NameError: name 'radmodel' is not defined

In [28]:
radmodel.OLR

NameError: name 'radmodel' is not defined

In [29]:
radmodel.ASR - radmodel.OLR

NameError: name 'radmodel' is not defined

In [30]:
while np.abs(radmodel.ASR - radmodel.OLR) > 0.01:
    radmodel.step_forward()

NameError: name 'np' is not defined

In [31]:
#  Check the energy budget again
radmodel.ASR - radmodel.OLR

NameError: name 'radmodel' is not defined

In [32]:
def add_profile(skew, model, linestyle='-', color=None):
    line = skew.plot(model.lev, model.Tatm - climlab.constants.tempCtoK,
             label=model.name, linewidth=2)[0]
    skew.plot(1000, model.Ts - climlab.constants.tempCtoK, 'o', 
              markersize=8, color=line.get_color())
    skew.ax.legend()

In [33]:
skew = make_skewT()
add_profile(skew, radmodel)
skew.ax.set_title('Pure radiative equilibrium', fontsize=18);

NameError: name 'plt' is not defined

In [34]:
# Make an exact clone of our existing model
radmodel_noH2O = climlab.process_like(radmodel)
radmodel_noH2O.name = 'Radiation (no H2O)'
print(radmodel_noH2O)

NameError: name 'climlab' is not defined

In [35]:
#  Here is the water vapor profile we started with
radmodel_noH2O.specific_humidity

NameError: name 'radmodel_noH2O' is not defined

In [36]:
radmodel_noH2O.specific_humidity *= 0.

NameError: name 'radmodel_noH2O' is not defined

In [37]:
radmodel_noH2O.specific_humidity

NameError: name 'radmodel_noH2O' is not defined

In [38]:
#  it's useful to take a single step first before starting the while loop
#   because the diagnostics won't get updated 
#  (and thus show the effects of removing water vapor)
#  until we take a step forward
radmodel_noH2O.step_forward()
while np.abs(radmodel_noH2O.ASR - radmodel_noH2O.OLR) > 0.01:
    radmodel_noH2O.step_forward()

NameError: name 'radmodel_noH2O' is not defined

In [39]:
radmodel_noH2O.ASR - radmodel_noH2O.OLR

NameError: name 'radmodel_noH2O' is not defined

In [40]:
skew = make_skewT()
for model in [radmodel, radmodel_noH2O]:
    add_profile(skew, model)

NameError: name 'plt' is not defined